# Transformer Models (Applications):

Transformer models like GPT,BERT etc.. are widely used in various NLP tasks, including text classification, named entity recognition, sentiment analysis, question answering, and machine translation.

## BERT (Bidirectional Encoder Representations from Transformers):
--------------

BERT is a pre-trained transformer model developed by Google, known for achieving state-of-the-art performance across a wide range of Natural Language Processing (NLP) tasks.

It leverages bidirectional context and self-attention mechanisms to capture
rich semantic representations of text data.


### BERT (Architecture):
-------------

BERT follows the architecture of transformer models, comprising multiple layers of encoders with self attention mechanisms. Unlike traditional models, BERT processes entire input sequences bidirectionally, allowing each token to attend to all other tokens in the sequence.

BERT is classified into two types:

● BERTBASE, BERTLARGE  based on the number of encoder layers, self-attention heads and hidden vector size.


### BERT (Pre-training Process):
-------
BERT undergoes two-stage pre-training on large text corpora:

#### 1. Masked Language Model (MLM):

   
   During pre-training, a certain percentage of input tokens (15%)  are randomly selected for masking . However not all simply replaced with [MASK]. Instead they are handled in three different ways: 80% of selected tokens are replaced with [MASK] , 10% of selected tokens replaces with a random wron word , 10% of selected tokens kapt as original word , and the model is trained to predict these masked tokens based on the context provided by surrounding tokens. This objective encourages BERT to understand bidirectional context and relationships between words.

   Since BERT's self-attention has no causal mask, every token — including [MASK] itself — can freely attend to every other token in both directions simultaneously.

   ![MLM example](Screenshot(104).png)

#### 2. Next Sentence Prediction (NSP):

 BERT is also trained on a second objective — learning whether two sentences are naturally connected or completely unrelated. This helps BERT understand discourse-level coherence and relationships between sentences, going beyond word-level understanding into document-level structure.

BERT receives a pair of sentences separated by a [SEP] token. 50% of the time, sentence B genuinely follows sentence A in the original document. The other 50% of the time, sentence B is a randomly sampled sentence from a completely different document. BERT must predict which case it is.


   In NSP : two sentences, in random order, are separated by the **[SEP]** token, for example:

   - **[CLS]** Toast is a simple yet delicious food **[SEP]** It’s often served with butter, jam, or honey.

label: isNext (so return true)  

   - **[CLS]** It’s often served with butter, jam, or honey. **[SEP]** The unemployment rate rose sharply last quarter.

label: NotNext (so return false)

   The **[CLS]** token is a **placeholder** token for the model, prompting the model to return a **True** or **False** label indicating whether the sentences topically coherent and contextually connected to each other

   


### BERT (Key Features ):
- **Bidirectional Context**:

  
    BERT captures bidirectional context effectively, enabling deeper understanding of word semantics and relationships.

- **Transfer Learning**:

  
    Pre-trained BERT models can be fine-tuned on downstream NLP tasks with task-specific data, allowing for efficient transfer learning and adaptation to diverse applications.


- **Contextual Representations**:

    BERT generates contextual representations for each token in the input sequence, which are rich in semantic information and contextually relevant to the surrounding tokens.


-----------------

### BERT (Advantages):

- **State-of-the-Art Performance**: BERT achieves state-of-the-art results on various NLP benchmarks, including text classification, named entity recognition, question answering, and sentiment analysis.

- **Language Agnostic**: BERT can be applied to multiple languages and domains without significant modifications, making it highly versatile and widely applicable.


- **Flexible Fine-Tuning**: Fine-tuning BERT on specific tasks requires minimal task-specific data and training time, enabling rapid development of high-performance NLP models.



-------------------------------------

## Calculating  **Semantic text similarity** using BERT:

In [5]:
!pip install transformers

In [4]:
!pip install torch torchvision torchaudio

`BertTokenizer`: Converts text into tokens (words/subwords) and then into numerical IDs that BERT understands.
`BertModel`: The actual BERT neural network that generates embeddings.

`bert-base-uncased`: A specific BERT model (12 layers, 110M parameters) that treats uppercase and lowercase as the same.

`from_pretrained()`: Downloads the pre-trained model and tokenizer from Hugging Face.

--------------------

You can download this models from here: (https://huggingface.co/google-bert/bert-base-uncased/tree/main):
download the following files and organize them in single folder: `config.json`, `model.safetensors`, `tokenizer.json`, `vocab.txt`, `tokenizer_config.json`.

- model.safetensors
This is the actual BERT model — all the learned weights from pre-training. The 110 million parameters across all 12 encoder layers, attention heads, and feed-forward networks. Without this, you have no model — just an empty architecture

- .safetensors is a newer, safer format than the old .bin format — it loads faster and prevents arbitrary code execution during loading.

- config.json
This tells the code what architecture to build before loading the weights.Without this, the code doesn't know whether it is loading BERT-base or BERT-large, how many layers to construct, what hidden size to expect.

- vocab.txt
This is BERT's vocabulary — all 30,522 tokens BERT knows about, one per line. 

- tokenizer.json
This is the full tokenizer saved in a single file — it contains the vocabulary, the merging rules, and all special token definitions ([CLS], [SEP], [MASK], [PAD], [UNK]) in one place. 

- tokenizer_config.json
This stores the tokenizer's settings and behavior


In [2]:
from transformers import BertTokenizer, BertModel
import torch
from scipy.spatial.distance import cosine

# Load pre-trained BERT model and tokenizer
local_model_path ="./models/bert-base-uncased-model"
tokenizer = BertTokenizer.from_pretrained(local_model_path)
model = BertModel.from_pretrained(local_model_path)

# Define input texts
text1 = "The quick brown fox jumps over the lazy dog."
text2 = "A fast red fox leaps across the sleepy canine."

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: ./models/bert-base-uncased-model
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [3]:

# Define input texts
text1 = "The quick brown fox jumps over the lazy dog."
text2 = "A fast red fox leaps across the sleepy canine."

`return_tensors='pt'`: Returns PyTorch tensors (not Python lists)

This creates token IDs, attention masks, etc., ready for the model.

In [4]:
# Tokenize the input texts
inputs1 = tokenizer(text1, return_tensors='pt')
inputs2 = tokenizer(text2, return_tensors='pt')

`torch.no_grad()`: Disables gradient calculation (saves memory since we're not training, just doing inference).

`model(**inputs1)`: Passes tokenized inputs to BERT.

In [5]:
# Get BERT embeddings (no gradient computation for efficiency)
# sinc we just want to get embeddings — we don't need gradients at all. 
# torch.no_geard() tell pyTorch :
# "Don't track anything, don't build a computation graph,
# just run the forward pass and give me the output."
with torch.no_grad():
    outputs1 = model(**inputs1)
    outputs2 = model(**inputs2)

pooler_output: A special 768-dimensional vector that represents the whole sentence (derived from the **[CLS]** token after processing through BERT). This is different from raw token embeddings.

- text1: "The quick brown fox jumps over the lazy dog."
  
           ↓ tokenizer
  
- [CLS] The quick brown fox jumps over the lazy dog . [SEP]

  Tokens: input_ids (example) = [101, 1996, 4248, 2829, 4419, 14523, 2058, 1996, 13971, 3899, 1012, 102]

  
           ↓ model(**inputs1)

  
- Each token → 768-dimensional contextual embedding
[CLS]  →  pooler_output  →  single sentence vector [1, 768]

In [7]:
# Get the pooled output embeddings
embed1 = outputs1.pooler_output
embed2 = outputs2.pooler_output
print(embed1)

tensor([[-8.2316e-01, -4.7679e-01, -8.8916e-01,  5.7508e-01,  7.1672e-01,
         -6.2258e-02,  8.4884e-01,  3.0954e-01, -6.9906e-01, -9.9999e-01,
         -3.3085e-01,  9.1141e-01,  9.9064e-01,  3.6720e-01,  9.4275e-01,
         -6.2431e-01, -2.3156e-01, -5.5003e-01,  1.7503e-01, -3.9238e-01,
          7.1458e-01,  9.9998e-01,  1.7245e-01,  2.6804e-01,  5.2037e-01,
          9.8159e-01, -7.1134e-01,  9.2269e-01,  9.5042e-01,  8.0547e-01,
         -4.7902e-01,  2.2420e-01, -9.9266e-01, -1.7688e-01, -8.7938e-01,
         -9.9019e-01,  4.0555e-01, -6.8363e-01,  2.7680e-02, -2.2669e-02,
         -8.9090e-01,  2.8895e-01,  9.9997e-01,  1.5186e-01,  4.9414e-01,
         -3.9523e-01, -1.0000e+00,  3.3622e-01, -8.7573e-01,  9.1648e-01,
          7.9610e-01,  8.4510e-01, -2.1564e-02,  4.9385e-01,  5.0232e-01,
         -1.2664e-01, -3.8033e-02,  8.2830e-02, -2.6015e-01, -5.9695e-01,
         -5.7473e-01,  3.3516e-01, -8.4686e-01, -8.0504e-01,  9.2370e-01,
          6.2531e-01, -1.2533e-01, -2.

`.detach()`: Removes the tensor from PyTorch's computation graph

`.numpy()`: Converts to a NumPy array for use with SciPy functions

In [13]:
# Convert PyTorch tensors to numpy arrays
embed1 = embed1.detach().numpy()
embed2 = embed2.detach().numpy()

Cosine distance: Measures the angle between two vectors (0 = identical direction, 2 = opposite).

Cosine similarity = 1 - cosine_distance (ranges from -1 to 1, where 1 means nearly identical meaning).

In [14]:
# Calculate cosine similarity
similarity = 1 - cosine(embed1[0], embed2[0])

print(f"Similarity between '{text1}' and '{text2}': {similarity}")

Similarity between 'The quick brown fox jumps over the lazy dog.' and 'A fast red fox leaps across the sleepy canine.': 0.9846379160881042


---------------------

## GPT (Generative Pre-trained Transformer):

GPT is a cutting-edge natural language processing (NLP) model developed by OpenAI. It revolutionizes language generation and comprehension tasks.  
Pretraining: GPT learns from vast amounts of text data to generate human-like responses.

### How GPT Works:

- Transformer Architecture: GPT employs a transformer architecture with attention mechanisms.
-  Contextual Understanding: It understands text in context, capturing nuances and semantics.
-  Autoregressive Generation: GPT generates text word by word, predicting the next word based on the preceding context.


### Autoregressive Generation:

- Autoregressive Model: GPT generates text sequentially, predicting the next word based on the preceding context.
- Probability Distribution: At each step, GPT computes the probability distribution over the vocabulary to select the most likely next word.
- Sampling Strategies: GPT can use various sampling strategies, such as greedy decoding or temperature-based sampling, to control the diversity and randomness of generated text.


### GPT:Training Process:

1.  Pre-training: GPT is initially pre-trained on a large corpus of text data using unsupervised learning objectives, such as language modeling.
2.  Fine-tuning: After pre-training, GPT can be fine-tuned on specific tasks using supervised learning with task-specific data and objectives.
3.  Transfer Learning: GPT's pre-trained weights can be transferred and fine-tuned on a wide range of downstream tasks with minimal task-specific data.



-----------------------

## Text Generation example using  distilGPT2:

`GPT2LMHeadModel`: GPT-2 with a language modeling head (predicts next words)

`GPT2Tokenizer`: Converts text to/from tokens (numbers the model understands)

`distilgpt2`: a smaller version of `gpt2` (only ~80 MB instead of 500 MB)

_______________

You can download this models from here: (https://huggingface.co/distilbert/distilgpt2/tree/main): download the following files and organize them in single folder: `config.json`, `generation_config_for_text_generation.json`, `model.safetensors`, `tokenizer.json`, `vocab.json`, `merges.txt`,  `tokenizer_config.json`, `generation_config.json`.

- config.json
Same as BERT — defines the model architecture before loading weights.
for example:

{n_layer:6, ...}

- model.safetensors
Same as BERT — the actual learned weights. All ~80 million parameters of distilGPT2.

- tokenizer.json
Same concept as BERT — the full tokenizer in one file. But GPT2 uses a completely different tokenization algorithm called Byte Pair Encoding (BPE) instead of BERT's WordPiece.

- tokenizer_config.json
Same as BERT — stores tokenizer settings and behavior like model max length and special token definitions.

- vocab.json
This is GPT2's equivalent of BERT's vocab.txt — but in JSON format instead of plain text.


- merges.txt
This is the core of BPE tokenization — something BERT does not have at all. It contains a list of rules for how to merge character pairs into subword tokens


- generation_config.json
This is completely absent in BERT — because BERT cannot generate text. This file stores the default generation settings for the mode
example:

{
  "eos_token_id": 50256, ...}




In [18]:
# Use DistilGPT-2 (only ~300 MB instead of 560 MB)
from transformers import GPT2LMHeadModel, GPT2Tokenizer

local_model_path = './models/distilgpt2_model'  # smaller than gpt2 !
tokenizer = GPT2Tokenizer.from_pretrained(local_model_path)
model = GPT2LMHeadModel.from_pretrained(local_model_path)


Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

In [19]:
# Define Input Prompt
prompt = "Once upon a time, there was a" # This is the starting text that the model will continue

# Encoding the Prompt
# Converts text into token IDs (numbers)
input_ids = tokenizer.encode(prompt, return_tensors='pt') # return_tensors='pt' returns PyTorch tensors


`max_length=100`: Total length (prompt + generated tokens) in tokens

`do_sample=True`: Random sampling (vs always picking most likely next word)

This is the switch between two completely different generation modes:

do_sample = False => greedy encoding, always picks the single highest probabitlity token (dereministic - same output every time) 

do_sample = True => random sampling , picks tokens based on their proability distrubtion , different output every run

`top_k=50`: Only consider the 50 most likely next tokens at each step (only works if do_sample = true)

`top_p=0.95`: Nucleus sampling: consider smallest set of tokens with 95% cumulative probability (only works if do_sample = ture)

`num_return_sequences=3`: Generate 3 different completions

In [20]:
# Generating Text
output = model.generate(input_ids, max_length=100, do_sample=True,
                        top_k=50, top_p=0.95, num_return_sequences=3)


[transformers] The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
[transformers] Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
[transformers] The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.




`skip_special_tokens=True` removes tokens like `<|endoftext|>`

In [21]:
# Converts token IDs back to readable text
for output_sequence in output:
    print(tokenizer.decode(output_sequence, skip_special_tokens=True))

Once upon a time, there was a moment in the universe. In this moment, we saw the stars as if they were nothing more than a galaxy full of stars.


"Starlight will be shining into space... and now, the cosmos will finally witness the passing of our solar system."
A team of astronomers from the Harvard School of Public Health and the Duke University School of Medicine were able to observe two large galaxies in the galactic sky at the same time. They saw the stars
Once upon a time, there was a huge opportunity to develop a brand new product and we saw it in stores and by doing that we opened that new business."
Once upon a time, there was a time when we could do it without feeling like we could do it without feeling like we could do it without feeling like we could do it without feeling like we could do it without feeling like we could do it without feeling like we could do it without feeling like we could do it without feeling like we could do it without feeling like we could do it witho